In [ ]:
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from pysheds.grid import Grid
from pysheds.sview import Raster
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from shapely.geometry import shape
from scipy.ndimage import gaussian_filter, distance_transform_edt
from rasterio.features import shapes
from tqdm import tqdm

In [ ]:
watershed = gpd.read_file("GIS/WATERSHEDS.geojson")
watershed = watershed[watershed['NAME'] == 'Popes Head Creek']
watershed = watershed.to_crs('EPSG:26918')

In [ ]:
# load elevation data for particular watershed
with rasterio.open("elevation/fairfax_dem_1m.tif") as src:    
    out_image, out_transform = mask(src, watershed.geometry, crop=True)
    out_meta = src.meta.copy()
    out_meta.update({
        "height": out_image.shape[1],
        "width": out_image.shape[2],
        "transform": out_transform
    })

with rasterio.open("elevation/popes_head_dem.tif", 'w', **out_meta) as dest:
    dest.write(out_image)

In [ ]:
grid = Grid.from_raster("elevation/popes_head_dem.tif")
dem = grid.read_raster("elevation/popes_head_dem.tif")

In [ ]:
plt.imshow(np.where(dem == -9999, np.nan, dem), cmap='terrain')
plt.title('Popes Head Creek DEM')
plt.colorbar(label='Elevation (m)')
plt.axis('off')
plt.show()

In [ ]:
pit_filled = grid.fill_pits(dem)
flooded = grid.fill_depressions(pit_filled)
inflated = grid.resolve_flats(flooded)
flowdir = grid.flowdir(inflated)
acc = grid.accumulation(flowdir)

In [ ]:
# with rasterio.open('flowdir.tif', "w",
#     driver="GTiff",
#     height=flowdir.shape[0],
#     width=flowdir.shape[1],
#     count=1,
#     dtype=flowdir.dtype,
#     crs=src.crs,
#     transform=grid.affine) as dst:
#     dst.write(flowdir, 1)

flowdir = grid.read_raster('flowdir.tif')

In [ ]:
acc_smooth = np.log10(acc+1)
acc_smooth = gaussian_filter(acc_smooth, sigma=3)
plt.imshow(acc_smooth)
plt.axis('off')
plt.colorbar(label='log2(Accumulation + 1)')
plt.title('Popes Head Creek Smoothed Accumulation')
plt.show()

In [ ]:
plt.imshow(np.where(dem == -9999, np.nan, flowdir))
plt.axis('off')
plt.colorbar(label='Flow Direction')
plt.show()

## Subcatchment Creation Using Recorded Inlets
Directly apply the recorded stormwater infrastructure data to identify points of urban inflow.

In [ ]:
infrastructure = gpd.read_file('stormnet/13.geojson')
infrastructure = infrastructure.to_crs(watershed.crs).clip(watershed)

In [ ]:
drainage_sources = ['Inlet', 'Infall']
is_source = [infrastructure['TYPE'].str.contains(source) for source in drainage_sources]
is_source = np.logical_or(*is_source)
is_source = np.where(is_source.isna(), False, is_source)
drainage = infrastructure[is_source]

## Identify Land Where Drainage Occurs

In [ ]:
subcatchments = np.zeros_like(acc)
mask = np.where(dem == -9999, True, False)
area = np.sum(~mask)
subcatchment_area = 0
geometries = tqdm(drainage.geometry)

sink_patch = np.array([
    [  2,   4,   8  ],  # NW, N, NE
    [ 1,  -2,  16 ],  # W,  PIT,  E
    [128,  64,  32 ]   # SW, S, SE
])

for source in geometries:
    j, i = grid.affine.__invert__()*(source.x, source.y)
    i, j = round(i), round(j)
    
    # manipulate inlet surroundings for water-intake emphasis (better delineation)
    original = flowdir[i-1:i+2, j-1:j+2].copy()
    flowdir[i-1:i+2, j-1:j+2] = sink_patch

    sub = grid.catchment(x=j, y=i, fdir=flowdir, xytype='index')
    flowdir[i-1:i+2, j-1:j+2] = original
    sub = np.where(mask, False, sub)
    mask[i, j] = True
    if not sub.sum():
        continue
    subcatchment_area += sub.sum()
    subcatchments[sub] = 1
    mask = np.logical_or(mask, sub)
    geometries.set_description(f"Area covered - {subcatchment_area/area*100:.2f}%")
    

In [ ]:
polygons = []
arr = (subcatchments != 0).astype(np.uint8)
for geom, val in shapes(arr, mask=arr==1, transform=grid.affine):
    polygons.append(shape(geom))

In [ ]:
polygon_gdf = gpd.GeoDataFrame(
    {"geometry": polygons},
    crs=grid.crs
)

## Delineate drainage land for subcatchments by infalls

In [ ]:
nodes = gpd.read_file('nodes.geojson')
edges = gpd.read_file('edges.geojson')

In [ ]:
infalls = nodes[nodes.node_type == 'infall'].clip(polygon_gdf)
infall_indices = []
for geom in infalls.geometry:
    j, i = ~grid.affine * (geom.x, geom.y)
    i, j = int(round(i)), int(round(j))
    infall_indices.append((i, j))

In [ ]:
infall_mask = np.zeros_like(subcatchments, dtype=bool)

for i, j in infall_indices:
    infall_mask[i, j] = True

distance, nearest_idx = distance_transform_edt(
    ~infall_mask,  # distance to nearest infall
    return_indices=True
)

assigned = np.zeros_like(subcatchments, dtype=int)
for idx, (i, j) in tqdm(list(enumerate(infall_indices, start=1))):
    nearest_i, nearest_j = nearest_idx
    assigned[(nearest_i == i) & (nearest_j == j) & (subcatchments==1)] = idx

In [ ]:
polygons = []
for label in tqdm(np.unique(assigned[assigned>0])):
    sub = (assigned == label).astype(np.uint8)
    for geom, val in shapes(sub, mask=sub>0, transform=grid.affine):
        polygons.append({
            "geometry": shape(geom)
        })

catchment_gdf = gpd.GeoDataFrame(polygons, crs=grid.crs)

In [ ]:
subcatchment_gdf = catchment_gdf[catchment_gdf.area >= 100] # ARBITRARY: minimum coverage of 100 m^2
infalls = infalls.clip(subcatchment_gdf)
subcatchment_gdf = gpd.sjoin(subcatchment_gdf, infalls, predicate="covers").drop(columns='index_right').reset_index(drop=True).reset_index().rename(columns={'index': 'subcatchment_id'})

In [ ]:
impervious_surfaces = gpd.read_file("GIS/impervious_surfaces/2023_Countywide_Impervious.shp").to_crs(grid.crs).clip(watershed)

In [ ]:
subcatchment_gdf['geometry'] = subcatchment_gdf.geometry.buffer(0)
impervious_surfaces['geometry'] = impervious_surfaces.geometry.buffer(0)
subcatchment_gdf['pct_impervious'] = subcatchment_gdf.geometry.apply(lambda geom: round(impervious_surfaces.clip(geom).area.sum() / geom.area * 100, 3))

In [ ]:
soils = gpd.read_file('GIS/SOIL.geojson').to_crs(grid.crs).clip(watershed)

In [ ]:
soils['HYDRO_GROUP'] = soils['HYDRO_GROUP'].replace(['C/D', 'B/D'], 'D').replace('NA', 'C').fillna('C')
green_ampt_params = {
    "A":  {"Ksat": 40, "Suction": 60,  "IMD": 0.30},
    "B":  {"Ksat": 20, "Suction": 110, "IMD": 0.35},
    "C":  {"Ksat": 6,  "Suction": 150, "IMD": 0.40},
    "D":  {"Ksat": 1,  "Suction": 200, "IMD": 0.45}
}
soils[['Ksat', 'Suction', 'IMD']] = soils['HYDRO_GROUP'].apply(
    lambda g: pd.Series(green_ampt_params[g])
)

In [ ]:
joined = gpd.overlay(subcatchment_gdf, soils, how='intersection')
joined['area'] = joined.geometry.area
grouped = joined.groupby('subcatchment_id').apply(
    lambda df: pd.Series({
        "Ksat": (df["area"] * df["Ksat"]).sum() / df["area"].sum(),
        "Suction": (df["area"] * df["Suction"]).sum() / df["area"].sum(),
        "IMD": (df["area"] * df["IMD"]).sum() / df["area"].sum(),
    })
)
subcatchment_gdf = subcatchment_gdf.merge(grouped, on='subcatchment_id')

In [ ]:
# find the mean slope of every subcatchment to its drainage point
for i, sub in tqdm(subcatchment_gdf.iterrows()):
    subset = nodes.clip(sub.geometry)
    max_diff = subset.elevation.max() - subset.elevation.min()
    subcatchment_gdf.loc[i, 'slope'] = max(max_diff / (sub.geometry.area ** 0.5) * 100, 0.1) # minimum slope of 0.1%

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
subcatchment_gdf.plot(ax=ax, column='pct_impervious', legend=True, edgecolor='black', linewidth=0.5)
infalls.plot(ax=ax, color='red', markersize=0.5)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Subcatchments of Popes Head Creek')
plt.show()

In [ ]:
subcatchment_gdf.to_file("subcatchments.geojson", driver="GeoJSON")